## RFE - recursive feature elimination   --  Classification 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
# from sklearn.feature_selection import SelectKBest
# from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pickle

# algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


In [2]:
# functions 
# feature selection
def rfe_features_classify(indep_x,dep_y,n):
    # model list 
    log_model=LogisticRegression(solver='liblinear', max_iter=1000)
    # knn_model=KNeighborsClassifier(n_neighbors=5,p=2, metric='minkowski')
    # nb_model=GaussianNB()
    # svc_model=SVC()
    dt_model=DecisionTreeClassifier(criterion='gini', splitter='best',random_state=0)
    rf_model=RandomForestClassifier(n_estimators=5, criterion='gini',random_state=0)
    rfe_model_list=[log_model  ,dt_model , rf_model]
    # rfe select features 
    rfe_result_list=[]
    for model in rfe_model_list:
        # print(model)
        rfe_alg=RFE(estimator=model,n_features_to_select=n,step=1)
        rfe_feature=rfe_alg.fit(indep_x,dep_y)
        rfe_features=rfe_feature.transform(indep_x)
        rfe_result_list.append(rfe_features)
        print('\n\nmodel: ',model, 'rfe_feature array: ',rfe_features)
        print('selected features: ', indep_x.columns[rfe_feature.get_support()])
    return rfe_result_list  

# tran test split and standardization 
def split_scaler(indep_x,dep_y):
    x_train,x_test,y_train,y_test = train_test_split(indep_x,dep_y,test_size=0.20,random_state=0)
    # standardization
    sc=StandardScaler()
    x_train=sc.fit_transform(x_train)
    x_test=sc.transform(x_test)
    return  x_train,x_test,y_train,y_test

# model prediction & evaluation 
def cm_pred_eval(classifier,x_test):
    # prediction
    y_pred=classifier.predict(x_test)
    # confusion matrix
    from sklearn.metrics import confusion_matrix
    cm=confusion_matrix(y_test,y_pred)
    # classification report 
    from sklearn.metrics import classification_report
    classi_report=classification_report(y_test,y_pred)
    # accuracy 
    from sklearn.metrics import accuracy_score
    accuracy=accuracy_score(y_test,y_pred)
    return classifier,accuracy,cm,classi_report,x_test,y_test

### --------------- classification algortihms -----------------------------
# logistic regression 
def logistic(x_train,y_train,x_test):
    # model creation
    from sklearn.linear_model import LogisticRegression
    classifier=LogisticRegression(random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# KNN 
def knn(x_train,y_train,x_test):
    # model creation
    from sklearn.neighbors import KNeighborsClassifier
    classifier=KNeighborsClassifier(n_neighbors=5,p=2, metric='minkowski')
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# SVM - linear 
def svm_linear(x_train,y_train,x_test):
    # model creation
    from sklearn.svm import SVC
    classifier=SVC(kernel='linear',random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# SVM - Non linear 
def svm_nonlinear(x_train,y_train,x_test):
    # model creation
    from sklearn.svm import SVC
    classifier=SVC(kernel='rbf',random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# navie bayes
def naive(x_train,y_train,x_test):
    # model creation 
    from sklearn.naive_bayes import GaussianNB
    classifier=GaussianNB()
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# decision tree
def decision_tree(x_train,y_train,x_test):
     # model creation 
    from sklearn.tree import DecisionTreeClassifier
    classifier=DecisionTreeClassifier(criterion='entropy',random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# random forest
def random_forest(x_train,y_train,x_test):
     # model creation 
    from sklearn.ensemble import RandomForestClassifier
    classifier=RandomForestClassifier(n_estimators=10,criterion='entropy',random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test


## ------------ final table for comparision 
def rfe_classification(rfe_resut_list):
    dataframe = pd.DataFrame(rfe_resut_list, index=['log_model','dt_model','rf_model'])
    dataframe
    return dataframe

In [17]:
data_original = pd.read_csv('preprocessed_CKD.csv')
data_ckd= data_original
data_ckd

data_ckd=pd.get_dummies(data_ckd,drop_first=True,dtype=int)
data_ckd.head()

# ip op split 
indep_x=data_ckd.drop('classification_notckd',axis=1)
dep_y = data_ckd['classification_notckd']

# k best 
no_of_features = 7
rfe_list=rfe_features_classify(indep_x,dep_y,no_of_features)
rfe_list

acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

rfe_resut_list=[]
for rfe_array in rfe_list:
    x_train,x_test,y_train,y_test=split_scaler(rfe_array,dep_y)   # (selected important features , target variable) 
    rfe_row = {} 
    # algorithms 
    classifier,accuracy,cm,classi_report,x_test,y_test=logistic(x_train,y_train,x_test)
    rfe_row['logistic']=accuracy
    classifier,accuracy,cm,classi_report,x_test,y_test=knn(x_train,y_train,x_test)
    rfe_row['knn']=accuracy
    classifier,accuracy,cm,classi_report,x_test,y_test=svm_linear(x_train,y_train,x_test)
    rfe_row['svm_linear']=accuracy
    classifier,accuracy,cm,classi_report,x_test,y_test=svm_nonlinear(x_train,y_train,x_test)
    rfe_row['svm_nonlinear']=accuracy
    classifier,accuracy,cm,classi_report,x_test,y_test=naive(x_train,y_train,x_test)
    rfe_row['naive']=accuracy
    classifier,accuracy,cm,classi_report,x_test,y_test=decision_tree(x_train,y_train,x_test)
    rfe_row['decision_tree']=accuracy
    classifier,accuracy,cm,classi_report,x_test,y_test=random_forest(x_train,y_train,x_test)
    rfe_row['random_forest']=accuracy
    rfe_resut_list.append(rfe_row)

rfe_final_result = rfe_classification(rfe_resut_list)



model:  LogisticRegression(max_iter=1000, solver='liblinear') rfe_feature array:  [[1.  1.2 0.  ... 0.  1.  1. ]
 [4.  0.8 0.  ... 0.  0.  0. ]
 [2.  1.8 1.  ... 0.  0.  1. ]
 ...
 [0.  0.6 1.  ... 0.  0.  0. ]
 [0.  1.  1.  ... 0.  0.  0. ]
 [0.  1.1 1.  ... 0.  0.  0. ]]
selected features:  Index(['al', 'sc', 'rbc_normal', 'rbc_unknown', 'pc_unknown', 'htn_yes',
       'dm_yes'],
      dtype='object')


model:  DecisionTreeClassifier(random_state=0) rfe_feature array:  [[ 1.02  36.     1.2   ... 44.     5.2    1.   ]
 [ 1.02  18.     0.8   ... 38.     0.     0.   ]
 [ 1.01  53.     1.8   ... 31.     0.     0.   ]
 ...
 [ 1.02  26.     0.6   ... 49.     5.4    0.   ]
 [ 1.025 50.     1.    ... 51.     5.9    0.   ]
 [ 1.025 18.     1.1   ... 53.     6.1    0.   ]]
selected features:  Index(['sg', 'bu', 'sc', 'hemo', 'pcv', 'rc', 'htn_yes'], dtype='object')


model:  RandomForestClassifier(n_estimators=5, random_state=0) rfe_feature array:  [[ 1.02   1.     1.2   ... 44.     5.2    0

In [18]:
rfe_final_result  #7

,logistic,knn,svm_linear,svm_nonlinear,naive,decision_tree,random_forest
log_model,0.9250,0.9875,1.0000,0.9375,0.9625,1.0,0.9875
dt_model,0.9875,1.0000,0.9875,1.0000,0.9250,1.0,1.0000
rf_model,0.9750,0.9875,0.9750,0.9750,0.9875,1.0,0.9875


In [16]:
rfe_final_result  #6

,logistic,knn,svm_linear,svm_nonlinear,naive,decision_tree,random_forest
log_model,0.9125,0.9875,0.9750,0.9375,0.9500,0.9875,0.95
dt_model,0.9875,1.0000,0.9875,1.0000,0.9250,1.0000,1.00
rf_model,0.9750,0.9750,0.9750,0.9750,0.9875,1.0000,1.00


In [4]:
rfe_final_result #5

,logistic,knn,svm_linear,svm_nonlinear,naive,decision_tree,random_forest
log_model,0.9125,0.9375,0.9250,0.9375,0.9500,0.9375,0.9500
dt_model,0.9875,1.0000,0.9875,1.0000,0.8875,1.0000,1.0000
rf_model,0.9750,0.9625,0.9750,0.9625,0.9375,0.9875,0.9625


In [12]:
rfe_final_result  #4

,logistic,knn,svm_linear,svm_nonlinear,naive,decision_tree,random_forest
log_model,0.9125,0.9375,0.9125,0.9375,0.950,0.9375,0.9500
dt_model,0.9875,1.0000,0.9875,1.0000,0.875,1.0000,1.0000
rf_model,0.9750,0.9750,0.9875,0.9750,0.950,0.9625,0.9625


In [14]:
rfe_final_result  #3

,logistic,knn,svm_linear,svm_nonlinear,naive,decision_tree,random_forest
log_model,0.9250,0.9250,0.9375,0.9375,0.9125,0.9375,0.9375
dt_model,0.9875,0.9875,0.9875,0.9875,0.8500,1.0000,1.0000
rf_model,0.9500,0.9375,0.9375,0.9125,0.9125,0.9750,0.9750


## RFE - recursive feature elimination  --   Regression  

In [8]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, chi2, RFE, RFECV
import pickle
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor


In [20]:
# functions 
# ------------------------------- functions  -------------------------------
# feature selection 
def rfe_regressor_feature(indep_x,dep_y,n):
    lr=LinearRegression()
    # svr=SVR(kernel='linear')
    dt=DecisionTreeRegressor(random_state=0)
    rf=RandomForestRegressor(n_estimators=10,random_state=0)
    rfe_model_list=[lr,dt,rf]
    # rfe feature selection 
    rfe_features_list=[]
    for model in rfe_model_list:
        rfe = RFE(estimator=model,n_features_to_select=n,step=1)
        rfe_fit=rfe.fit(indep_x,dep_y)
        rfe_features=rfe_fit.transform(indep_x)
        rfe_features_list.append(rfe_features)
    return rfe_features_list

# regression model predict & evaluation 
def r2_score_pred(regressor,x_test,y_test):
    y_pred=regressor.predict(x_test)
    from sklearn.metrics import r2_score
    r2_score=r2_score(y_test,y_pred)
    return r2_score 

# tran test split and standardization 
def split_scaler(indep_x,dep_y):
    x_train,x_test,y_train,y_test = train_test_split(indep_x,dep_y,test_size=0.20,random_state=0)
    # standardization
    sc=StandardScaler()
    x_train=sc.fit_transform(x_train)
    x_test=sc.transform(x_test)
    return  x_train,x_test,y_train,y_test
    
# regression algorithms 
# linear
def linear(x_train,y_train,x_test):
    from sklearn.linear_model import LinearRegression
    regressor=LinearRegression()
    regressor.fit(x_train,y_train)
    r2 = r2_score_pred(regressor,x_test,y_test)
    return r2 
    
# SVR -linear
def svrl(x_train,y_train,x_test):
    from sklearn.svm import SVR
    regressor=SVR(kernel='linear')
    regressor.fit(x_train,y_train)
    r2 = r2_score_pred(regressor,x_test,y_test)
    return r2 
    
# SVR-non linear
def svrnl(x_train,y_train,x_test):
    from sklearn.svm import SVR
    regressor=SVR(kernel='rbf')
    regressor.fit(x_train,y_train)
    r2 = r2_score_pred(regressor,x_test,y_test)
    return r2 
    
# decision tree
def decisionReg(x_train,y_train,x_test):
    from sklearn.tree import DecisionTreeRegressor
    regressor=DecisionTreeRegressor(criterion='squared_error',random_state=0)
    regressor.fit(x_train,y_train)
    r2 = r2_score_pred(regressor,x_test,y_test)
    return r2 
    
# random forest 
def randomForestReg(x_train,y_train,x_test):
    from sklearn.ensemble import RandomForestRegressor
    regressor=RandomForestRegressor(n_estimators=10,random_state=0)
    regressor.fit(x_train,y_train)
    r2 = r2_score_pred(regressor,x_test,y_test)
    return r2 


# select features table 
def rfe_regression(rfe_result_list):
    dataframe=pd.DataFrame(rfe_result_list,index=['linear','DT','RF'])
    return dataframe


In [27]:
# main program

# main program 
data_original = pd.read_csv('preprocessed_CKD.csv')
data_ckd= data_original
data_ckd

data_ckd=pd.get_dummies(data_ckd,drop_first=True,dtype=int)
data_ckd.head()

# ip op split 
indep_x=data_ckd.drop(['classification_notckd'],axis=1)
dep_y = data_ckd['classification_notckd']

# feature selection 
no_of_features= 3
rfe=rfe_regressor_feature(indep_x,dep_y,no_of_features)


rfe_result_list=[]

for features in rfe:
    x_train,x_test,y_train,y_test=split_scaler(features,dep_y)

    rfe_dict={}
    # algorithms 
    r2=linear(x_train,y_train,x_test)
    rfe_dict['linear']=r2
    r2=svrl(x_train,y_train,x_test)
    rfe_dict['svrl']=r2
    r2=svrnl(x_train,y_train,x_test)
    rfe_dict['svrnl']=r2
    r2=decisionReg(x_train,y_train,x_test)
    rfe_dict['decision tree']=r2
    r2=randomForestReg(x_train,y_train,x_test)
    rfe_dict['random forest']=r2

    rfe_result_list.append(rfe_dict)


result_regression = rfe_regression(rfe_result_list)


In [24]:
result_regression   #6

,linear,svrl,svrnl,decision tree,random forest
linear,0.624867,0.529910,0.719281,0.749811,0.747500
DT,0.818199,0.811570,0.963261,0.945055,0.982418
RF,0.735048,0.728223,0.895379,1.000000,0.937912


In [22]:
result_regression   #5

,linear,svrl,svrnl,decision tree,random forest
linear,0.624867,0.529979,0.733426,0.749811,0.747500
DT,0.816796,0.814783,0.970781,1.000000,0.994505
RF,0.810116,0.815751,0.963664,1.000000,0.991834


In [26]:
result_regression   #4

,linear,svrl,svrnl,decision tree,random forest
linear,0.624867,0.529979,0.735349,0.749811,0.748469
DT,0.817905,0.821866,0.979503,0.994798,0.988802
RF,0.809360,0.813085,0.969088,0.943012,0.984554


In [28]:
result_regression   #3

,linear,svrl,svrnl,decision tree,random forest
linear,0.519542,0.291523,0.619198,0.589117,0.587261
DT,0.696984,0.687656,0.939484,0.991209,0.993951
RF,0.752168,0.738763,0.963120,0.998338,0.990996


I experimented with Recursive Feature Elimination on my regression dataset, testing feature subsets from 3 to 6 features.

With 3 features, most models’ accuracy dropped significantly, except for Random Forest and Decision Tree.
With 4 features, the performance improved slightly but still lacked consistency across all models.
At 5 features, most models including linear, SVR, Decision Tree, and Random Forest performed well, with R² or accuracy ranging from 0.7–1.0, indicating a balanced performance across models.
With 6 features, some models like Decision Tree achieved near-perfect performance, but others did not improve substantially, suggesting potential overfitting.

Based on this analysis, I concluded that 5 features provide the optimal balance between predictive power and generalization, making it the most reliable choice for regression modeling.